# 03 · Model evaluation, revisited

Notebook 02 reports **R² = 0.9997** for a Random Forest that predicts a metro's home value (ZHVI). This notebook explains why that score is inflated and re-evaluates the models properly.

**What changes here**
1. **Leak fixed.** In notebook 02, `ZHVI_roll3` averages the current month with the two before it, so the target is part of the inputs.
2. **Time-based split.** Models train on 2000–2019 and are tested on 2020–2025, as a real forecast would be.
3. **Real baseline.** Linear Regression is fitted and scored, not just described.
4. **Harder, more useful task.** Besides predicting next month, the models predict prices **12 months ahead** using only data available a year earlier.

MAPE (mean absolute percentage error) is reported alongside R², because R² across ~900 metros of very different price levels looks high even for weak models.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/reshaped_zillow.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["RegionName", "Date"]).dropna(subset=["ZHVI"])
print(f"{len(df):,} monthly rows · {df.RegionName.nunique()} metros · {df.Date.min():%b %Y} – {df.Date.max():%b %Y}")

220,722 monthly rows · 894 metros · Jan 2000 – Feb 2025


## 1. Showing the leak

With notebook 02's definition, `ZHVI_roll3 = (ZHVI_t + ZHVI_t-1 + ZHVI_t-2) / 3`. Rearranged, the target can be computed exactly from three of the inputs:

In [2]:
g = df.groupby("RegionName")["ZHVI"]
roll3_original = g.transform(lambda x: x.rolling(3).mean())
reconstructed = 3 * roll3_original - g.shift(1) - g.shift(2)
gap = (reconstructed - df["ZHVI"]).abs().dropna()
print(f"Largest difference between the reconstructed and the actual ZHVI: {gap.max():.6f}")

Largest difference between the reconstructed and the actual ZHVI: 0.000000


A difference of zero means the model can read the answer from its inputs. The fix is to build the rolling average from **past months only** (`shift(1)` before `rolling(3)`).

## 2. Features

In [3]:
for k in (1, 2, 3, 12, 13, 14):
    df[f"lag{k}"] = g.shift(k)
df["roll3_past"] = g.transform(lambda x: x.shift(1).rolling(3).mean())      # fixed: past months only
df["roll3_year_ago"] = g.transform(lambda x: x.shift(12).rolling(3).mean())  # for 12-month-ahead
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df = pd.get_dummies(df, columns=["State"], drop_first=True)
state_cols = [c for c in df.columns if c.startswith("State_")]

setups = {
    "1 month ahead":   ["lag1", "lag2", "lag3", "roll3_past", "SizeRank", "Year", "Month"],
    "12 months ahead": ["lag12", "lag13", "lag14", "roll3_year_ago", "SizeRank", "Month"],
}

## 3. Train on 2000–2019, test on 2020–2025

The training set is sampled to 40,000 rows (fixed seed) to keep the Random Forest quick; every test row is used.

In [4]:
rows = []
for setup, feats in setups.items():
    d = df.dropna(subset=["ZHVI"] + feats)
    train = d[d["Year"] <= 2019].sample(n=40000, random_state=42)
    test = d[d["Year"] >= 2020]
    X = feats + state_cols
    for name, model in [("Linear Regression", LinearRegression()),
                        ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))]:
        model.fit(train[X], train["ZHVI"])
        pred = model.predict(test[X])
        rows.append({
            "Task": setup, "Model": name,
            "R²": round(r2_score(test["ZHVI"], pred), 4),
            "MAE ($)": round(mean_absolute_error(test["ZHVI"], pred)),
            "RMSE ($)": round(np.sqrt(mean_squared_error(test["ZHVI"], pred))),
            "MAPE (%)": round(np.mean(np.abs(pred - test["ZHVI"]) / test["ZHVI"]) * 100, 2),
            "Test rows": len(test),
        })
results = pd.DataFrame(rows)
results

,Task,Model,R²,MAE ($),RMSE ($),MAPE (%),Test rows
0,1 month ahead,Linear Regression,1.0000,619,1065,0.26,55417
1,1 month ahead,Random Forest,0.9938,1865,12698,0.45,55417
2,12 months ahead,Linear Regression,0.9795,12749,23075,4.84,55411
3,12 months ahead,Random Forest,0.9711,13834,27411,5.02,55411


## 4. What the results mean

- **Next-month prediction is easy** even without the leak: home values move slowly, so last month's value is an excellent guess. Near-perfect scores here say little about forecasting skill.
- **12 months ahead is the meaningful test.** Both models stay within about **5% of the actual price on average** for 2020–2025, a period that includes the post-COVID price surge.
- **Linear Regression edges out Random Forest** a year ahead. Tree models cannot predict values above the range they were trained on, and post-2020 prices rose past 2000–2019 levels in many metros.

**Bottom line:** the original 0.9997 reflects a leaked feature and a random split. The corrected one-year-ahead error of about 5% is the honest measure of the model's forecasting accuracy.